# Demo: `LinearOperator`

https://github.com/cornellius-gp/linear_operator

`pip install git+https://github.com/cornellius-gp/linear_operator.git`

*Assumption we've made so far:* Matrices are (with a few exceptions) symmetric positive definite. This can and should be relaxed for a general library.

In [102]:
import torch 

from linear_operator.operators import DiagLinearOperator, BlockDiagLinearOperator, KroneckerProductLinearOperator

In [179]:
def make_pd(n, b=tuple()):
    a = torch.rand(*b, n, n)
    return a @ a.transpose(-1, -2) + torch.diag_embed(torch.rand(*b, n))

### Simple example: Diagonal matrices

Consider diagonal matrices of size $n \times n$

Matmul is $\mathcal O(n)$ using structure, but $\mathcal O(n^2)$ not using structure. Same for memory complexity.

In [6]:
diag1 = torch.rand(5000)
diag2 = torch.rand(5000)

Diag1 = diag1.diag() # 25M elements
Diag2 = diag2.diag() # 25M elements

Diag1_lo = DiagLinearOperator(diag1)  # 5K elements
Diag2_lo = DiagLinearOperator(diag2)  # 5K elements

#### Addition

Diagonality (ness?) is closed under addition. `LinearOperator` understands that (note that the result is again a `DiagLinearOperator` rather than a dense Tensor).

In [34]:
Diag1_lo + Diag2_lo

DiagLinearOperator of dimension (5000, 5000) at 0x7fcdd0c9e100

#### Matmul

Matrix-multiplying diagonal matrices just means creating a diagonal matrix with the element-wise product of the diagonals as its diagonal. Time / mem complexity naive: $\mathcal O(n^2)$, time /mem complexity using structure: $\mathcal O(n)$

In [17]:
matmul = (Diag1 @ Diag2).diag()
matmul_lo = (Diag1_lo @ Diag2_lo).diag()

torch.allclose(matmul, matmul_lo)

True

In [10]:
%timeit (Diag1 @ Diag2).diag()

2.33 s ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%timeit (Diag1_lo @ Diag2_lo).diag()

29.3 µs ± 264 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


Improvements: 
- $5,000$ -fold reduction in memory
- $\approx 80,000$ times faster

#### Symeig

This uses `__torch_function__` in order to dispatch `torch.symeig` to a custom implementation that essentially just returns the diagonal elements and the identity matrix (should sort the evals and permute the evecs to have the exact same behavior, that's an easy thing to do).

Time complexity goes from $\mathcal O(n^3)$ to $\mathcal O(1)$, memory complexity from $\mathcal O(n^2)$ to $\mathcal O(n)$. 

Of course if the user was aware of the structure, they could do this manually. The point is that `LinearOperator` does these things automatically (for more complex examples see below). Think of it like operator fusing on steroids (with the steroids being exploiting linear algebra simplifications for structured operators - this is something that basic sparsity cannot do achieve).

In [25]:
evals, evecs = torch.symeig(Diag1)
evals_lo, evecs_lo = torch.symeig(Diag1_lo)

torch.allclose(evals, torch.sort(evals_lo).values)

True

In [26]:
%timeit torch.symeig(Diag1)

2.22 s ± 55.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%timeit torch.symeig(Diag1_lo)

2.88 µs ± 102 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


Improvements: 
- $5,000$ -fold reduction in memory
- $\approx 1,000,000$ times faster

### Simpl-ish example: Block-Diagonal matrices

Matmul is O(n) using structure, but O(n^2) not using structure. Same for memory complexity.

In [180]:
# create a 2000 x 2000 block-diagonal matrix with 200 random symmetric 10 x 10 matrices on the (block)diagonal
BDiag_lo = BlockDiagLinearOperator(make_pd(10, (200,)))

# instatiate the full matrix
BDiag = BDiag_lo.to_dense()

#### MVM

In [181]:
v = torch.rand(2000, 1)

In [182]:
mvm = BDiag @ v
mvm_lo = BDiag_lo @ v

torch.allclose(mvm, mvm_lo)

True

In [183]:
%timeit BDiag @ v

525 µs ± 14.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [184]:
%timeit BDiag_lo @ v

107 µs ± 1.21 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


Improvements: 
- $2,000$ -fold reduction in memory
- $\approx 4$ times faster (dense matmuls are just really optimized...)

#### SVD

Can construct the SVD of a Kronecker product from the SVD of the constitutent matrices. This allows us to compute the SVDs of 200 10x10 matrices in batch under the hood rather than the SVD of a 2000 x 2000 matrix.

In math, since time complexity for computing SVDs is cubic, if there are $n_b$ blocks of size $n \times n$ in the matrix, then we reduce complexity from $\mathcal O (n_b^3 n^3)$ to $\mathcal O (n_b n^3)$

In [185]:
U, S, V = torch.svd(BDiag)
U_lo, S_lo, V_lo = torch.svd(BDiag_lo)

torch.allclose(S, torch.sort(S_lo, descending=True).values, atol=1e-5)

True

In [186]:
%timeit torch.svd(BDiag)

1.08 s ± 25.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [187]:
%timeit torch.svd(BDiag_lo)

3.14 µs ± 120 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


Improvements: 
- $\approx 333,000$ times faster  (a lot more than the $200^2$ we'd expect in asymptopia)

### More complex: Kronecker matrices

If $A$ is $n \times n$ and $B$ is $m \times m$, then $A\otimes B$ is $nm \times nm$

In [188]:
A = make_pd(20)
B = make_pd(500)

Kron_lo = KroneckerProductLinearOperator(A, B)
Kron = Kron_lo.to_dense()

torch.allclose(Kron, torch.kron(A, B))

True

#### MVM

Naively, MVM is $\mathcal O(n^2m^2)$ time. However, exploiting Kronecker structure, we get $\mathcal O (nm (n+m))$ time. Memory complexity goes from $\mathcal O(n^2m^2)$ to $\mathcal O(n^2 + m^2)$

In [189]:
v = torch.rand(Kron_lo.shape[-1], 1)

In [190]:
mvm = Kron @ v
mvm_lo = Kron_lo @ v

torch.allclose(mvm, mvm_lo)

True

In [191]:
%timeit Kron @ v

19.8 ms ± 770 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [192]:
%timeit Kron_lo @ v

200 µs ± 10.3 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


Improvements:
- $200,000$ -fold reduction in memory
- $\approx 100$ times faster

# There is other cool stuff

But some of the best is currently only implemented in `gpytorch`'s `LazyTensor` module (just need to move over and rename...)

`pip install git+https://github.com/cornellius-gp/gpytorch.git`

In [193]:
from gpytorch.lazy import KroneckerProductLazyTensor, DiagLazyTensor, ConstantDiagLazyTensor

In [194]:
Kron_lt = KroneckerProductLazyTensor(A, B)

Let's add some (constant) diagonal: 

In [195]:
Diag_lt = ConstantDiagLazyTensor(1 + torch.rand(1), Kron_lt.shape[-1])
Diag = Diag_lt.evaluate()

KaddD = Kron + Diag
KaddD_lt = Kron_lt + Diag_lt
KaddD_lt

In [196]:
torch.allclose(KaddD_lt.evaluate(), KaddD)

True

#### Solve

Solving $(A \otimes B + a I)x = v$ naively means solving a $nm \times nm$ linear system.

We can be smart by instead noting that computing the inverse of $A \otimes B + a I$ can be done cheaply:
1. We perform an eigendecomposition $A \otimes B = \sum_j e_j v_jv_j^T$. This can be done cheaply b/c the eigendecomposition of  $A \otimes B$ can be constructed from the (small and cheap-to-compute) eigendecompositions of $A$ and $B$, respectively.
2. The eigendecomposition of $A \otimes B + a I$ is just the eigendecomposition of $A \otimes B$ plus a spectral shift of the eigenvalues $e_j$ by $a$.
3. The inverse of $A \otimes B + a I$ is obtained by simply taking the reciprocals of the eigenvalues in its eigendecomposition.

At the end of the day, this means that we can go from $\mathcal O(n^3m^3)$ to $\mathcal O(n^3 + m^3)$ complexity for the solve. We don't have to do anything other than ensuring that we express $A \otimes B$ and the constant diagonal with the right operators, the rest we get for free (modulo registering this with `solve` via `__torch_function__`). Of course this is true for additional Kronecker factors, i.e. we go from $\mathcal O(\Pi_i n_i^3)$ to $\mathcal O(\sum_i n_i^3)$

One thing to be careful about are numerical robustness issues, which can be an issue when computing eigendecompositions. In general, hairy linear algebra should happen in double not float...

In [197]:
x = torch.linalg.solve(KaddD, v)
x_lt = KaddD_lt.inv_matmul(v)  # TODO: Use __torch_function__ to dispatch torch.linalg.solve to this method

In [201]:
torch.allclose(x, x_lt, atol=1e-3)

True

In [202]:
%timeit torch.linalg.solve(Kron + Diag, v)

6.27 s ± 105 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [203]:
%timeit (Kron_lt + Diag_lt).inv_matmul(v)

38 ms ± 1.88 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Improvements:
- $200,000$ -fold reduction in memory
- $\approx 165$ times faster

#### Triangular 

We have `torch.triangular_solve` to get solve complexity down from $\mathcal O(n^3)$ to $\mathcal O(n^2)$, but the user has to fully trace through all of their code to understand when it's safe to call it. If we can retain the structural information, we can just dispatch to the right solve automatically, allowing us to write structure-agnostic code and get the linear algebra optimziations for free.

In [208]:
from linear_operator.operators import TriangularLinearOperator

In [262]:
tri = torch.eye(500) + torch.rand(500, 500, dtype=torch.double).tril()
tri_lo = TriangularLinearOperator(tri)

torch.equal(tri, tri_lo.to_dense())

True

In [266]:
tri_inv = torch.inverse(tri)
tri_lo_inv = tri_lo.inverse()  # TODO: Handle in torch.inverse by registering via __torch_function__

torch.allclose(tri_inv, tri_lo_inv.to_dense())

True

In [264]:
%timeit torch.inverse(tri)

6.46 ms ± 231 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [265]:
%timeit tri_lo.inverse()

1.3 µs ± 41.8 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


Improvements:
- uses half of the memory
- $\approx 5,000$ times faster